In [1]:
# Automatically reload .py files when they are changed
%load_ext autoreload
%autoreload 2

# Numerical packages
import numpy as np
from scipy import optimize

# Plotting
import matplotlib.pyplot as plt

plt.rcParams.update({
    'axes.grid': True,
    'grid.color': 'black',
    'grid.alpha': 0.25,
    'grid.linestyle': '-'
})
plt.rcParams.update({'font.size': 14})

colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

# Import the model classes
from Consumer import ConsumerClass
from Government import GovernmentClass

print("Everything is set up correctly!")

Everything is set up correctly!


In [2]:
model = ConsumerClass()

print(model)

ConsumerClass
  alpha = 0.6000, beta = 0.5000
  sigma_A = 0.8000, sigma_B = 0.4000
  p1 = 1.0000, p2 = 1.0000, p3 = 1.5000
  I = 10.0000


In [3]:
opt_grid = model.solve_grid(N=100)

s1 = 0.535354
w  = 0.444444

Budget shares:
s1 = 0.535354
s2 = 0.206510
s3 = 0.258137

utility = 3.401482
function evaluations = 10,000


In [4]:
opt = model.solve()

success = True
message = CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL

Nested shares:
s1 = 0.535623
w  = 0.439479

Budget shares:
s1 = 0.535623
s2 = 0.204084
s3 = 0.260293
sum = 1.000000

utility = 3.401680
iterations = 4
function evaluations = 18


In [5]:
# Check that the numerical solution satisfies the basic properties of the model

# a. collect the three optimal budget shares
shares = np.array([
    opt.s1,
    opt.s2,
    opt.s3
])

# b. all budget shares should be strictly between zero and one
assert np.all(shares > 0.0), 'at least one budget share is not positive'
assert np.all(shares < 1.0), 'at least one budget share is not below one'

# c. the budget shares should sum to exactly one
assert np.isclose(
    np.sum(shares),
    1.0
), 'budget shares do not sum to one'

# d. translate the optimal shares into quantities
x1,x2,x3 = model.quantities(
    opt.s1,
    opt.w
)

quantities = np.array([
    x1,
    x2,
    x3
])

# e. all quantities should be positive
assert np.all(
    quantities > 0.0
), 'at least one optimal quantity is not positive'

# f. grid search and L-BFGS-B should give approximately the same solution
# The grid solution is only approximate because N = 100 is relatively coarse.
assert np.allclose(
    [opt_grid.s1,opt_grid.w],
    [opt.s1,opt.w],
    atol=0.01
), 'grid search and L-BFGS-B give very different solutions'

print('all checks passed')

all checks passed


## 2. Solving the consumer problem numerically


### 2.1 Two-dimensional grid search

I first solve the consumer problem using a two-dimensional grid search over the nested budget shares $s_1$ and $w$. I compare different grid sizes to examine the trade-off between numerical accuracy and computational cost. I use the L-BFGS-B solution as a benchmark because it is not restricted to discrete grid points.

In [6]:
# I import the packages I need for timing and presenting the results
import time
import pandas as pd


# a. I solve the model with L-BFGS-B and use this solution as my benchmark
opt_lbfgsb = model.solve(
    do_print=False
)


# b. I choose the grid sizes I want to compare
N_values = [
    50,
    100,
    500,
    1000
]


# c. I create an empty list in which I store the results from each grid search
grid_results = []


# d. I solve the model for each grid size
for N in N_values:

    # i. I start the timer before solving the model
    t0 = time.perf_counter()

    # ii. I solve the consumer problem using an N x N grid
    opt_grid_N = model.solve_grid(
        N=N,
        do_print=False
    )

    # iii. I calculate how long the grid search took
    elapsed_time = time.perf_counter() - t0

    # iv. I calculate the distance between the grid solution
    # and the L-BFGS-B solution in the two choice variables
    distance = np.sqrt(
        (opt_grid_N.s1 - opt_lbfgsb.s1)**2
        +
        (opt_grid_N.w - opt_lbfgsb.w)**2
    )

    # v. I calculate how much lower utility is relative to the
    # more precise L-BFGS-B solution
    utility_gap = opt_lbfgsb.u - opt_grid_N.u

    # vi. I store the results for this grid size
    grid_results.append({
        'N': N,
        's1': opt_grid_N.s1,
        'w': opt_grid_N.w,
        'utility': opt_grid_N.u,
        'distance_to_L-BFGS-B': distance,
        'utility_gap': utility_gap,
        'function_evaluations': opt_grid_N.nfev,
        'time_seconds': elapsed_time
    })


# e. I collect the results in a DataFrame to make the comparison easy to read
grid_table = pd.DataFrame(
    grid_results
)


# f. I display the comparison table
grid_table

,N,s1,w,utility,distance_to_L-BFGS-B,utility_gap,function_evaluations,time_seconds
0,50,0.530612,0.448980,3.400748,0.010741,9.321692e-04,2500,0.000490
1,100,0.535354,0.444444,3.401482,0.004973,1.977149e-04,10000,0.000772
2,500,0.535070,0.438878,3.401674,0.000817,5.510127e-06,250000,0.024723
3,1000,0.535536,0.439439,3.401680,0.000096,7.844534e-08,1000000,0.089119
